# 소형 Transformer 모델의 가중치를 2개의 GPU로 분리하여 Forward Pass 계산 후 검증

> 본 노트북은 SYLLABUS.md에 기반하여 자동 생성된 뼈대(Skeleton) 파일입니다. 상세한 이론, 수식 및 코드는 추가로 구현되어야 합니다.

## 1. 개요 및 학습 목표
이 노트북에서는 해당 주제에 대한 핵심 개념을 다룹니다.

## 2. 핵심 이론 및 수학적 원리
- 수식 및 상세한 동작 원리를 여기에 기록합니다.

## 3. 실습 코드 구현
아래 셀을 통해 파이썬 및 관련 프레임워크 코드를 직접 작성하고 실행해 볼 수 있습니다.

In [ ]:
# 실습을 위한 기본 라이브러리 임포트
import tensorflow as tf
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")


---
## Q1: 행렬 Column-Parallel 분할

텐서 병렬화(Tensor Parallelism)에서 Linear 레이어의 가중치 행렬을 **열(Column) 방향**으로 분할하는 방식을 구현하세요.

**요구사항:**
- 가중치 행렬 `W = np.random.randn(768, 3072)`를 `num_gpus=4`로 열 방향으로 균등 분할하세요.
- 각 GPU 파티션 `W_col[i]`의 shape을 출력하고, 입력 `x (shape: [2, 768])`와 행렬 곱 결과의 shape을 확인하세요.
- 4개 GPU 결과를 `np.concatenate`로 합산(All-Gather)하면 단일 GPU 결과와 동일한지 검증하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import numpy as np

np.random.seed(42)
num_gpus = 4
batch_size, d_model, d_ff = 2, 768, 3072

W = np.random.randn(d_model, d_ff)  # (768, 3072)
x = np.random.randn(batch_size, d_model)  # (2, 768)

# Column-Parallel 분할
col_per_gpu = d_ff // num_gpus
W_col = [W[:, i*col_per_gpu:(i+1)*col_per_gpu] for i in range(num_gpus)]

partial_outs = [x @ W_col[i] for i in range(num_gpus)]
all_gather_out = np.concatenate(partial_outs, axis=-1)  # All-Gather

# 검증
single_out = x @ W
print(f"분할 전 단일 GPU 결과 shape: {single_out.shape}")
print(f"All-Gather 후 결과 shape: {all_gather_out.shape}")
print(f"최대 오차: {np.max(np.abs(single_out - all_gather_out)):.2e}")
```
</details>

---
## Q2: Row-Parallel 분할 후 All-Reduce

가중치를 **행(Row) 방향**으로 분할하는 Row-Parallel 방식을 구현하고, 각 GPU의 부분 결과를 All-Reduce(합산)하여 최종 결과를 구하세요.

**요구사항:**
- `W = (d_ff, d_model)` 형태의 행렬을 행 방향으로 `num_gpus`등분하세요.
- 입력 `x (shape: [batch, d_ff])`도 같은 방식으로 분할하여 각 GPU에 배분하세요.
- 각 GPU의 부분 행렬 곱 결과를 `np.sum`으로 All-Reduce하고, 단일 GPU 결과와 비교하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import numpy as np

np.random.seed(0)
num_gpus = 4
batch_size, d_model, d_ff = 2, 768, 3072

W2 = np.random.randn(d_ff, d_model)  # (3072, 768)
x2 = np.random.randn(batch_size, d_ff)  # (2, 3072)

# Row-Parallel 분할
row_per_gpu = d_ff // num_gpus
W_row = [W2[i*row_per_gpu:(i+1)*row_per_gpu, :] for i in range(num_gpus)]
x2_parts = [x2[:, i*row_per_gpu:(i+1)*row_per_gpu] for i in range(num_gpus)]

partial_outs = [x2_parts[i] @ W_row[i] for i in range(num_gpus)]
all_reduce_out = np.sum(partial_outs, axis=0)  # All-Reduce

single_out = x2 @ W2
print(f"최대 오차: {np.max(np.abs(single_out - all_reduce_out)):.2e}")
print(f"Row-Parallel + All-Reduce 검증 성공: {np.max(np.abs(single_out - all_reduce_out)) < 1e-8}")
```
</details>

---
## Q3: 통신 비용 분석

Column-Parallel과 Row-Parallel 방식에서 각각 발생하는 통신의 종류와 크기를 계산하여 비교하세요.

**요구사항:**
- Column-Parallel의 All-Gather로 전달되는 총 데이터 크기(bytes)를 계산하세요.
- Row-Parallel의 All-Reduce로 전달되는 총 데이터 크기(bytes)를 계산하세요.
- 예: 데이터 타입 float32(4 bytes)를 기준으로 각 방식의 통신량을 출력하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
bytes_per_elem = 4  # float32
batch_size = 2
d_model = 768
d_ff = 3072

# All-Gather: 각 GPU가 자신의 출력 (batch, d_ff/num_gpus)를 보냄
# num_gpus-1번의 send 필요, 각 send = batch * (d_ff/num_gpus) elements
col_comm = (num_gpus - 1) * batch_size * (d_ff // num_gpus) * bytes_per_elem

# All-Reduce: 각 GPU가 결과 (batch, d_model)를 합산
row_comm = (num_gpus - 1) * batch_size * d_model * bytes_per_elem

print("통신 비용 비교 (forward pass, float32):")
print(f"  Column-Parallel All-Gather: {col_comm:,} bytes ({col_comm/1024:.2f} KB)")
print(f"  Row-Parallel All-Reduce:    {row_comm:,} bytes ({row_comm/1024:.2f} KB)")
print(f"\n비율: {col_comm/row_comm:.2f}x")
```
</details>